# Generating Graphs

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib import colormaps
from itertools import cycle
import os
import re

# Load Data

In [2]:
output_dir = "../graphs/ablation_modbertbr"
os.makedirs(output_dir, exist_ok=True)

filepaths = [
    "../data/eval/results_eval_mteb_v1_11_datasets_w_mteb.csv",
    "../data/eval/results_eval_mteb_default.csv",
    "../data/eval/results_eval_mteb_closed_source.csv"
    ]

filepath_finetuned_matryoshka = "../data/eval/results_eval_mteb_v1_11_datasets_w_mteb.csv"
filepath_baseline = "../data/eval/results_eval_mteb_default.csv"
# model_names = [
#     "e5-large-matryoshka-sts-pt",
#     "Legal-BERTimbau-sts-large-ma-v3",
#     "Qwen3-Embedding-0.6B",
#     "multilingual-e5-large"
#     ]#['BERTimbau-large-matryoshka-sts-pt', 'BERTimbau-large-sts-pt'] 
model_names = ['ModBERTBr-matryoshka-sts-pt', 'ModBERTBr-sts-pt'] 
# model_names_to_remove = [
#     'BERTimbau-large-sts-pt',
#     'e5-large-sts-pt',
#     'ModBERTBr-sts-pt',
#     'bert-base-portuguese-cased',
#     'mmBERT-base',
#     'ModBERTBr',
#     'bert-large-portuguese-cased',
#     'NeoBERTugues'
#     ] #, 'bert-base-portuguese-cased', 'mmBERT-base'
model_names_to_remove = []

results = [pd.read_csv(filepath) for filepath in filepaths]

In [3]:
results

[                                  model_name  embedding_dim  \
 0    iara-project/e5-large-matryoshka-sts-pt            NaN   
 1    iara-project/e5-large-matryoshka-sts-pt            NaN   
 2    iara-project/e5-large-matryoshka-sts-pt            NaN   
 3    iara-project/e5-large-matryoshka-sts-pt            NaN   
 4    iara-project/e5-large-matryoshka-sts-pt            NaN   
 ..                                       ...            ...   
 395            iara-project/ModBERTBr-sts-pt           64.0   
 396            iara-project/ModBERTBr-sts-pt           64.0   
 397            iara-project/ModBERTBr-sts-pt           64.0   
 398            iara-project/ModBERTBr-sts-pt           64.0   
 399            iara-project/ModBERTBr-sts-pt           64.0   
 
                           task_name  main_score  
 0                         Assin2STS    0.841035  
 1                       SICK-BR-STS    0.917959  
 2       STSBenchmarkMultilingualSTS    0.879975  
 3       MassiveIntentClas

# Pre-processing

In [4]:
task_type_map = {
    "Assin2STS": "STS",
    "SICK-BR-STS": "STS",
    "STSBenchmarkMultilingualSTS": "STS",
    'MassiveIntentClassification': "Classification",
    'MultiHateClassification': "Classification",
    'BrazilianToxicTweetsClassification': "Classification",
    'HateSpeechPortugueseClassification': "Classification",
    'TweetSentimentClassification': "Classification",
    # 'MultiLongDocReranking': "Reranking",
    'WikipediaRerankingMultilingual': "Reranking",
    'XGlueWPRReranking': "Reranking",
    'WebFAQRetrieval': "Retrieval",
    # 'MultiLongDocRetrieval': "Retrieval",
    'WikipediaRetrievalMultilingual': "Retrieval",
}

max_emb_dim_map = {
    'paraphrase-multilingual-MiniLM-L12-v2': 384,
    'BERTimbau': 1024,
    'Qwen3': 1024,
    'bert-base-portuguese-cased': 1024,
    'mmBERT': 768,
    'e5': 1024,
    'ModBERTBr': 768,
    'NeoBERTugues': 768,
    'Qwen3-Embedding-0.6B-sts-pt': 1024,
    'serafim': 1536,
    'bert-large': 1024,
    'gte-multilingual': 768
}

In [5]:
def map_model_name(model_raw_name: str, mapping: dict) -> int:
    matches = [
        value
        for key, value in mapping.items()
        if key.lower() in model_raw_name.lower()
    ]

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found for '{model_raw_name}': "
            f"{[k for k in mapping if k.lower() in model_raw_name.lower()]}"
        )
    if len(matches) == 0:
        return None  # ou raise ValueError se quiser exigir match obrigatório

    return matches[0]

In [6]:
results_all = pd.concat(results, ignore_index=True)
results_all['task_type'] = results_all['task_name'].map(task_type_map)
results_all.dropna(subset=['task_type'], inplace=True)
results_all['model_raw_name'] = results_all['model_name'].apply(lambda x: x.split("/")[-1])
results_all["max_emb_size"] = results_all["model_raw_name"].apply(
    lambda x: map_model_name(x, max_emb_dim_map)
)
results_all['embedding_dim_full'] = results_all['embedding_dim'].combine_first(results_all['max_emb_size']).astype(int)

if model_names:
    results_all = results_all.query("model_raw_name.isin(@model_names)")

if model_names_to_remove:
    results_all = results_all.query("~model_raw_name.isin(@model_names_to_remove)")

results_all.head()

,model_name,embedding_dim,task_name,main_score,task_key,task_type,model_raw_name,max_emb_size,embedding_dim_full
280,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,Assin2STS,0.818191,NaN,STS,ModBERTBr-matryoshka-sts-pt,768.0,768
281,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,SICK-BR-STS,0.916931,NaN,STS,ModBERTBr-matryoshka-sts-pt,768.0,768
282,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,0.821221,NaN,STS,ModBERTBr-matryoshka-sts-pt,768.0,768
283,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,MassiveIntentClassification,0.593341,NaN,Classification,ModBERTBr-matryoshka-sts-pt,768.0,768
284,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,MultiHateClassification,0.605000,NaN,Classification,ModBERTBr-matryoshka-sts-pt,768.0,768


In [7]:
results_all.task_name.unique()

<ArrowStringArray>
[                         'Assin2STS',                        'SICK-BR-STS',
        'STSBenchmarkMultilingualSTS',        'MassiveIntentClassification',
            'MultiHateClassification', 'BrazilianToxicTweetsClassification',
 'HateSpeechPortugueseClassification',       'TweetSentimentClassification',
     'WikipediaRerankingMultilingual',                  'XGlueWPRReranking',
                    'WebFAQRetrieval',     'WikipediaRetrievalMultilingual']
Length: 12, dtype: str

In [8]:
results_all.query("model_raw_name == 'ModBERTBr-matryoshka-sts-pt' & max_emb_size == embedding_dim_full").groupby("task_type")['main_score'].mean()

task_type
Classification    0.487510
Reranking         0.701763
Retrieval         0.595305
STS               0.852114
Name: main_score, dtype: float64

In [9]:
# cond_e5 = results_all['model_raw_name'].str.contains('e5')
# cond_bertimbau = results_all['model_raw_name'].str.contains('BERTimbau')
# cond_qwen = results_all['model_raw_name'].str.contains('Qwen')
# cond_mmbert_embed = results_all['model_raw_name'].str.contains('mmbert-embed')
# results_all = results_all.loc[cond_e5 | cond_bertimbau | cond_qwen | cond_mmbert_embed]
results_all.head()

,model_name,embedding_dim,task_name,main_score,task_key,task_type,model_raw_name,max_emb_size,embedding_dim_full
280,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,Assin2STS,0.818191,NaN,STS,ModBERTBr-matryoshka-sts-pt,768.0,768
281,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,SICK-BR-STS,0.916931,NaN,STS,ModBERTBr-matryoshka-sts-pt,768.0,768
282,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,0.821221,NaN,STS,ModBERTBr-matryoshka-sts-pt,768.0,768
283,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,MassiveIntentClassification,0.593341,NaN,Classification,ModBERTBr-matryoshka-sts-pt,768.0,768
284,iara-project/ModBERTBr-matryoshka-sts-pt,NaN,MultiHateClassification,0.605000,NaN,Classification,ModBERTBr-matryoshka-sts-pt,768.0,768


# Summarized Metrics

In [10]:
results_all.model_raw_name.unique()

<ArrowStringArray>
['ModBERTBr-matryoshka-sts-pt', 'ModBERTBr-sts-pt']
Length: 2, dtype: str

In [11]:
# baseline_model = 'Legal-BERTimbau-sts-large-ma-v3'
baseline_model = 'amazon.titan-embed-text-v2:0'
compare_model = 'e5-large-matryoshka-sts-pt'

In [12]:
results_max = results_all.sort_values(by = 'embedding_dim_full', ascending=False).groupby(['model_raw_name', 'task_name']).head(1)
grouped_results = results_max.query("model_raw_name.isin([@baseline_model, @compare_model])") \
                    .groupby(["model_raw_name", "task_type"])['main_score'] \
                    .mean() \
                    .reset_index()
grouped_results

,model_raw_name,task_type,main_score


In [13]:
def percent_diff_by_task_type(grouped_results, baseline_model, compare_model):
    wide = (
        grouped_results
        .pivot(index="task_type", columns="model_raw_name", values="main_score")
        .reindex(columns=[baseline_model, compare_model])
    )

    out = pd.DataFrame(index=wide.index)
    out["baseline"] = wide[baseline_model]
    out["compare"]  = wide[compare_model]
    out["abs_diff"] = out["compare"] - out["baseline"]

    # avoid division-by-zero issues
    out["pct_diff_vs_baseline"] = np.where(
        out["baseline"].ne(0),
        (out["abs_diff"] / out["baseline"]) * 100.0,
        np.nan
    )

    return out.reset_index().sort_values("task_type")

# usage
diff_table = percent_diff_by_task_type(grouped_results, baseline_model, compare_model)
diff_table

,task_type,baseline,compare,abs_diff,pct_diff_vs_baseline


In [14]:
df_results_ratio = pd.DataFrame()

results_all["num_dims"] = results_all["embedding_dim"].fillna(results_all["embedding_dim_full"])
results_all["num_dims"] = pd.to_numeric(results_all["num_dims"], errors="coerce")
df_ratio = results_all.query("max_emb_size > 768").copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce").astype(int)
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])

df_ratio = (
    df_ratio
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)

task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = df_tt[df_tt["num_dims"] == df_tt["max_dim"]][
        ["model_raw_name", "main_score"]
    ].rename(columns={"main_score": "max_score"})

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]

    df_results_ratio = pd.concat([df_results_ratio, df_tt], axis=0, ignore_index=True)
df_results_ratio

""


In [15]:
print(compare_model)
df_compare_ratio = df_results_ratio.query("model_raw_name == @compare_model & num_dims == 64")
print(f"Média Ratio: {df_compare_ratio['ratio'].mean()}")
df_compare_ratio

e5-large-matryoshka-sts-pt


UndefinedVariableError: name 'model_raw_name' is not defined

In [17]:
print(baseline_model)
df_compare_ratio = df_results_ratio.query("model_raw_name == @baseline_model & num_dims == 64")
print(f"Média Ratio: {df_compare_ratio['ratio'].mean()}")
df_compare_ratio

amazon.titan-embed-text-v2:0


UndefinedVariableError: name 'model_raw_name' is not defined

# Export Graphs

In [ ]:
# -------------------------
# Utils
# -------------------------
def sanitize_filename(name):
    name = str(name)
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.lower()


def apply_y_axis_formatting(
    ax,
    series,
    step=0.03,
    margin=0.01,
    y_bottom_override=None,  # force inferior y-limit if provided
    y_top_override=None,  # force inferior y-limit if provided
):
    """
    If y_bottom_override is set (e.g., 0.40), the y-axis lower bound will be forced
    to that value to create extra space at the bottom (e.g., for legends).
    """
    y_min = float(series.min())
    y_max = float(series.max())

    y_bottom = (np.floor(y_min / step) * step) - margin
    y_top = (np.floor(y_max / step) * step) + step + margin

    if y_bottom_override is not None:
        y_bottom = float(y_bottom_override)

    if y_top_override is not None:
        y_top = float(y_top_override)

    ax.set_ylim(y_bottom, y_top)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.tick_params(axis="y", labelsize=25)
    ax.tick_params(axis='x', labelsize=25, labelrotation=45)


def apply_log2_xaxis(ax, x_order, left_pad=1.08, right_pad=1.18):
    """
    Match the reference spacing (8,16,32,...) using log base 2.
    Adds a bit of right padding so end-labels have room.
    """
    x_order = sorted(pd.unique(pd.Series(x_order).dropna()))
    ax.set_xscale("log", base=2)
    ax.set_xticks(x_order)

    x_set = set([float(x) for x in x_order])

    def _fmt(x, pos):
        if float(x) in x_set:
            if abs(x - round(x)) < 1e-9:
                return str(int(round(x)))
            return f"{x:g}"
        return ""

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt))

    # Padding for labels on the right
    x_min = float(min(x_order))
    x_max = float(max(x_order))
    ax.set_xlim(x_min / left_pad, x_max * right_pad)


def _high_contrast_colors(n: int):
    """
    Use default Matplotlib colors first, then extend with Matplotlib categorical palettes.
    Designed for <= 13 models (your case), but remains robust if you exceed it.
    """
    n = int(n)
    if n <= 0:
        return []

    # 1) Default Matplotlib cycle (typically 10 colors: tab10)
    base = mpl.rcParams["axes.prop_cycle"].by_key().get("color", [])
    colors = [mpl.colors.to_hex(c) for c in base]

    if n <= len(colors):
        return colors[:n]

    # 2) Extend using additional built-in categorical palettes
    extra_sources = (
        list(colormaps["tab20"].colors)
        + list(colormaps["tab20b"].colors)
        + list(colormaps["tab20c"].colors)
    )

    seen = set(c.lower() for c in colors)
    for c in extra_sources:
        hx = mpl.colors.to_hex(c)
        if hx.lower() not in seen:
            colors.append(hx)
            seen.add(hx.lower())
        if len(colors) >= n:
            return colors[:n]

    # 3) Failsafe fallback (unlikely for your case)
    cmap = colormaps["turbo"]
    vals = np.linspace(0.0, 1.0, n, endpoint=True)
    return [mpl.colors.to_hex(cmap(v)) for v in vals]


def make_model_style_map(model_names):
    model_names = list(model_names)
    colors = _high_contrast_colors(len(model_names))

    markers = ["o", "s", "^", "v", "D", "P", "X", "<", ">", "h", "p", "8"]
    dash_patterns = [
        None,          # solid
        (6, 2),        # dashed
        (2, 2),        # densely dashed
        (6, 2, 1, 2),  # dash-dot-ish
        (1, 1),        # dotted
    ]

    style_map = {}
    for i, m in enumerate(model_names):
        style_map[m] = {
            "color": colors[i],
            "marker": markers[i % len(markers)],
            "dashes": dash_patterns[i % len(dash_patterns)],
        }
    return style_map


def build_custom_lineplot(
    ax,
    data,
    style_map,
    highlight_models=None,
    highlight_kwargs=None,
    other_kwargs=None,
    model_col="model_raw_name",  # allows plotting by display-name column
):
    """
    Emphasis strategy:
      - non-highlight models: lower alpha, thinner lines, smaller markers
      - highlighted models: thicker lines, bigger markers, higher zorder (drawn on top)
    Returns:
      - end_points: list of dicts with x,y,color for labels (ONLY highlight models by default)
    """
    highlight_models = set(highlight_models or [])

    highlight_kwargs = highlight_kwargs or dict(
        linewidth=5,
        markersize=10.0,
        alpha=1.0,
        zorder=6,
        markeredgewidth=1.0,
        markeredgecolor="white",
    )
    other_kwargs = other_kwargs or dict(
        linewidth=4,
        markersize=10.0,
        alpha=1.0,
        zorder=2,
        markeredgewidth=0.0,
    )

    end_points = []

    models_in_data = list(pd.unique(data[model_col]))
    draw_order = [m for m in models_in_data if m not in highlight_models] + [
        m for m in models_in_data if m in highlight_models
    ]

    for model in draw_order:
        df_m = data[data[model_col] == model].sort_values("num_dims")
        if df_m.empty:
            continue

        st = style_map.get(model, {})
        is_highlight = model in highlight_models
        kw = highlight_kwargs if is_highlight else other_kwargs

        (line,) = ax.plot(
            df_m["num_dims"].to_numpy(),
            df_m["main_score"].to_numpy(),
            label=model,
            color=st.get("color", None),
            marker=st.get("marker", "o"),
            linewidth=kw["linewidth"],
            markersize=kw["markersize"],
            alpha=kw["alpha"],
            zorder=kw["zorder"],
            markeredgewidth=kw.get("markeredgewidth", 0.0),
            markeredgecolor=kw.get("markeredgecolor", None),
        )
        if st.get("dashes") is not None:
            line.set_dashes(st["dashes"])

        if is_highlight:
            last = df_m.iloc[-1]
            end_points.append(
                dict(
                    model=model,
                    x=float(last["num_dims"]),
                    y=float(last["main_score"]),
                    color=st.get("color", "black"),
                )
            )

    return end_points


def annotate_end_labels_with_vertical_spacing(
    ax,
    end_points,
    x_mult=1.06,
    min_sep_frac=0.03,
    clamp=True,
):
    """
    Places labels near the last point of each line, but "repels" them vertically
    so they don't overlap.
    """
    if not end_points:
        return

    y0, y1 = ax.get_ylim()
    y_span = (y1 - y0) if (y1 - y0) != 0 else 1.0
    min_sep = y_span * float(min_sep_frac)

    pts = sorted(end_points, key=lambda d: d["y"])
    y_adj = []
    for i, d in enumerate(pts):
        y_target = d["y"]
        if i == 0:
            y_new = y_target
        else:
            y_new = max(y_target, y_adj[-1] + min_sep)
        y_adj.append(y_new)

    if clamp:
        overflow = y_adj[-1] - y1
        if overflow > -0.02:
            y_adj = [yy - overflow for yy in y_adj]
        underflow = y0 - y_adj[0]
        if underflow > -0.02:
            y_adj = [yy + underflow for yy in y_adj]

    for d, yy in zip(pts, y_adj):
        x_text = d["x"] * float(x_mult)
        ax.annotate(
            d["model"],
            xy=(d["x"], d["y"]),
            xytext=(x_text, yy),
            textcoords="data",
            fontsize=17,
            fontweight="bold",
            color=d["color"],
            va="center",
            ha="left",
            zorder=10,
            arrowprops=dict(arrowstyle="-", lw=0.8, color=d["color"], alpha=0.7),
        )


def fix_legend_inside(ax, highlight_models=None, preset="bottom_right"):
    """
    Keep legend inside axes. Reorder to show highlighted models first.

    preset:
      - "bottom_right" (default): inside, bottom-right
      - "upper_left": inside, upper-left
    """
    highlight_models = set(highlight_models or [])
    handles, labels = ax.get_legend_handles_labels()

    idx_high = [i for i, lab in enumerate(labels) if lab in highlight_models]
    idx_rest = [i for i, lab in enumerate(labels) if lab not in highlight_models]
    new_idx = idx_high + idx_rest

    handles = [handles[i] for i in new_idx]
    labels = [labels[i] for i in new_idx]

    if preset == "upper_left":
        loc = "upper left"
        bbox = (0.02, 0.98)
    else:  # "bottom_right"
        loc = "lower right"
        bbox = (0.98, 0.06)

    leg = ax.legend(
        handles,
        labels,
        loc=loc,
        bbox_to_anchor=bbox,
        frameon=True,
        framealpha=0.92,
        fontsize=12,
        borderaxespad=0.0,
        handlelength=3.0,
        labelspacing=0.35
    )
    leg.get_frame().set_linewidth(0.8)


# -------------------------
# Theme
# -------------------------
sns.set_theme(style="whitegrid", context="talk", font_scale=1.05)


# =========================
# 0) Model rename map
# =========================
MODEL_RENAME = {
    # Closed-source
    "global.cohere.embed-v4:0": "Cohere Embed v4",
    "amazon.titan-embed-text-v2:0": "Amazon Titan v2",
    "text-embedding-3-large": "Text Embedding 3 Large",
    # Peer-reviewed
    "serafim-900m-portuguese-pt-sentence-encoder": "Serafim 900m",
    "multilingual-e5-large": "Multilingual e5 Large",
    "Qwen3-Embedding-0.6B": "Qwen3 Embedding 0.6B",
    "qwen3-embedding-0.6b": "Qwen3 Embedding 0.6B",
    "gte-multilingual-base": "GTE Multilingual Base",
    "Legal-BERTimbau-sts-large-ma-v3": "Legal BERTimbau STS",
    "BERTimbau-large": "BERTimbau Large",
    "mmBERT-base": "mmBERT Base",
    "ModBERT-BR": "ModBERTBr",
    "ModBERTBr": "ModBERTBr",
    # Community
    "NeoBERTugues": "NeoBERTugues",
    "mmbert-embed-32k-2d-matryoshka": "mmBERT Embed 32k",
    # Fine-tuned (Ours)
    "e5-large-matryoshka-sts-pt": "e5 Large MRL",
    "BERTimbau-large-matryoshka-sts-pt": "BERTimbau Large MRL",
    "ModBERTBr-matryoshka-sts-pt": "ModBERTBr MRL",
}


# =========================
# 0.1) Per-task customizable y-bottom limits (NEW)
# =========================
# Use None to keep auto. Set any task you want to open space for legend.
Y_TOP_BY_TASK_NAME = {
    # Example:
    # "my_task_name": 0.40,
}

Y_TOP_BY_TASK_TYPE = {
    "Reranking": 0.91
}

Y_BOTTOM_BY_TASK_NAME = {
    # Example:
    # "my_task_name": 0.40,
}

Y_BOTTOM_BY_TASK_TYPE = {
    # Example:
    "STS": 0.65,
    "Classification": 0.43,
}

Y_BOTTOM_DEFAULT = None
Y_TOP_DEFAULT = None


# =========================
# 1) Preparação dos dados
# =========================
df_plot = results_all.copy()

df_plot["num_dims"] = df_plot["embedding_dim"].fillna(df_plot["embedding_dim_full"])
df_plot["num_dims"] = pd.to_numeric(df_plot["num_dims"], errors="coerce")
df_plot["main_score"] = pd.to_numeric(df_plot["main_score"], errors="coerce")

df_plot = df_plot.dropna(subset=["num_dims", "main_score", "model_raw_name"])
df_plot["model_raw_name"] = df_plot["model_raw_name"].astype(str)

# Display name used for legend + labels
df_plot["model_name"] = df_plot["model_raw_name"].replace(MODEL_RENAME)

# Build style map on display names (legend names)
model_order = sorted(df_plot["model_name"].unique())
MODEL_STYLE = make_model_style_map(model_order)

# Models to emphasize (and label at the end) — use DISPLAY NAMES
HIGHLIGHT_MODELS = {
    # Example:
    "BERTimbau Large MRL",
    "e5 Large MRL",
    "ModBERTBr MRL",
}


# =========================================================
# 2) Gráficos por task_name
# =========================================================
task_names = sorted(df_plot["task_name"].dropna().unique())
score_map = {
    'Classification': 'Accuracy',
    'STS': 'Spearman Corr.',
    'Retrieval': 'nDCG@10',
    'Reranking': 'MAP@1000'
}


for task in task_names:
    df_task = (
        df_plot[df_plot["task_name"] == task]
        .groupby(["task_name", "model_name", "num_dims"], as_index=False)["main_score"]
        .mean()
    )

    x_order = sorted(df_task["num_dims"].unique())

    fig, ax = plt.subplots(figsize=(11, 8))
    end_points = build_custom_lineplot(
        ax,
        df_task,
        MODEL_STYLE,
        highlight_models=HIGHLIGHT_MODELS,
        model_col="model_name",
    )

    ax.set_title(f"Task: {task}", fontsize=25, fontweight="bold", pad=10)
    ax.set_xlabel("Embedding Size", fontsize=20)
    ax.set_ylabel("Score", fontsize=20)

    apply_log2_xaxis(ax, x_order)

    y_override_bottom = Y_BOTTOM_BY_TASK_NAME.get(task, Y_BOTTOM_DEFAULT)
    y_override_top = Y_TOP_BY_TASK_NAME.get(task, Y_TOP_DEFAULT)
    apply_y_axis_formatting(
        ax,
        df_task["main_score"],
        step=0.03,
        margin=0.01,
        y_top_override=y_override_top,
        y_bottom_override=y_override_bottom,
    )

    annotate_end_labels_with_vertical_spacing(
        ax,
        end_points,
        x_mult=1.06,
        min_sep_frac=0.035,
        clamp=True,
    )

    # Keep the same behavior as before: bottom-right inside
    fix_legend_inside(ax, highlight_models=HIGHLIGHT_MODELS, preset="bottom_right")

    plt.tight_layout()

    filename = f"task_{sanitize_filename(task)}.pdf"
    fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
    plt.close(fig)


# =========================================================
# 3) Gráficos por task_type (média)
# =========================================================
df_task_type = df_plot.groupby(["task_type", "model_name", "num_dims"], as_index=False)["main_score"].mean()

task_types = sorted(df_task_type["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_task_type[df_task_type["task_type"] == ttype].copy()
    x_order = sorted(df_tt["num_dims"].unique())

    fig, ax = plt.subplots(figsize=(11, 8))
    end_points = build_custom_lineplot(ß
        ax,
        df_tt,
        MODEL_STYLE,
        highlight_models=HIGHLIGHT_MODELS,
        model_col="model_name",
    )

    # ax.set_title(f"Task Type: {ttype}", fontsize=30, fontweight="bold", pad=10)
    ax.set_xlabel("Embedding Size", fontsize=25)
    ax.set_ylabel(score_map[ttype], fontsize=25)

    apply_log2_xaxis(ax, x_order)

    y_override_bottom = Y_BOTTOM_BY_TASK_TYPE.get(ttype, Y_BOTTOM_DEFAULT)
    y_override_top = Y_TOP_BY_TASK_TYPE.get(ttype, Y_TOP_DEFAULT)

    apply_y_axis_formatting(
        ax,
        df_tt["main_score"],
        step=0.03,
        margin=0.01,
        y_top_override=y_override_top,
        y_bottom_override=y_override_bottom,
    )

    annotate_end_labels_with_vertical_spacing(
        ax,
        end_points,
        x_mult=1.06,
        min_sep_frac=0.035,
        clamp=True,
    )

    # NEW: Classification / Reranking -> upper-left inside; otherwise bottom-right inside
    legend_preset = "upper_left" if ttype in {"Classification", "Reranking"} else "bottom_right"
    fix_legend_inside(ax, highlight_models=HIGHLIGHT_MODELS, preset=legend_preset)

    plt.tight_layout()

    filename = f"task_type_{sanitize_filename(ttype)}.pdf"
    fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
    plt.close(fig)

In [ ]:
# =========================================================
# 5) Ratio vs full embedding por task_type
# =========================================================

map_lim_task = {
    "Classification": 0.8,
    "Reranking": 0.85,
    "Retrieval": 0.35,
    "STS": 0.9,
}

MODEL_RENAME = {
    'e5-large-matryoshka-sts-pt': 'e5 Large Matryoshka STS',
    'e5-large-sts-pt': 'e5 Large STS',
    'BERTimbau-large-matryoshka-sts-pt': 'BERTimbau Large Matryoshka STS',
    'BERTimbau-large-sts-pt': 'BERTimbau Large STS',
    'ModBERTBr-matryoshka-sts-pt': 'ModBERTBr Large Matryoshka STS',
    'ModBERTBr-sts-pt': 'ModBERTBr Large STS'
}

df_ratio = df_plot.copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce").astype("Int64")
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])
df_ratio["num_dims"] = df_ratio["num_dims"].astype(int)

df_ratio = (
    df_ratio.groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)


def get_hue_order(models):
    """
    Retorna a lista de modelos ordenada: primeiro os sem matryoshka,
    depois os com matryoshka — agrupando visualmente as barras.
    """
    standard = sorted([m for m in models if ("matryoshka" not in m.lower() and "qwen" not in m.lower())])
    matryoshka = sorted([m for m in models if ("matryoshka" in m.lower() or "qwen" in m.lower())])
    return standard + matryoshka


def apply_bar_hatch(ax):
    """
    Aplica hachura nos modelos com 'matryoshka' (ou 'qwen') e preenchimento
    sólido nos demais. Cria duas legendas no canto superior direito, dentro do gráfico.
    """
    handles, labels = ax.get_legend_handles_labels()

    # # Aplica hatch nas barras (cada container corresponde a um item do hue)
    # for container, label in zip(ax.containers, labels):
    #     hatch = "//" if ("matryoshka" in label.lower() or "qwen" in label.lower()) else ""
    #     for bar in container:
    #         bar.set_hatch(hatch)

    # # Espelha o hatch nos itens da legenda dos modelos
    # for handle, label in zip(handles, labels):
    #     hatch = "//" if ("matryoshka" in label.lower() or "qwen" in label.lower()) else ""
    #     try:
    #         handle.set_hatch(hatch)
    #     except Exception:
    #         pass

    # Remove qualquer legenda padrão criada pelo seaborn (vamos recriar)
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # --- Bloco 1: legenda dos modelos (cores + hatch), top-right (inside) ---
    leg1 = ax.legend(
        handles=handles,
        labels=labels,
        loc="lower right",
        fontsize=12,
        frameon=True,
        borderaxespad=0.4,
    )
    ax.add_artist(leg1)

    ax.tick_params(axis="y", labelsize=25)
    ax.tick_params(axis='x', labelsize=25, labelrotation=45)

    # # --- Bloco 2: legenda do tipo de preenchimento (abaixo), ainda top-right (inside) ---
    # solid_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="", label="Standard")
    # hatched_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="//", label="Matryoshka")

    # ax.legend(
    #     handles=[solid_patch, hatched_patch],
    #     title="Loss",
    #     loc="upper right",
    #     bbox_to_anchor=(1.0, 0.74),   # ajuste fino para ficar abaixo da leg1 (dentro do axes)
    #     bbox_transform=ax.transAxes,
    #     fontsize=9,
    #     title_fontsize=10,
    #     frameon=True,
    #     borderaxespad=0.4,
    # )


task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = (
        df_tt[df_tt["num_dims"] == df_tt["max_dim"]][["model_raw_name", "main_score"]]
        .rename(columns={"main_score": "max_score"})
    )

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]
    df_tt['model_raw_name'] = df_tt['model_raw_name'].map(MODEL_RENAME)

    hue_order = get_hue_order(df_tt["model_raw_name"].unique())

    fig, ax = plt.subplots(figsize=(5, 8))

    sns.barplot(
        data=df_tt,
        x="num_dims",
        y="ratio",
        hue="model_raw_name",
        hue_order=hue_order,
        ax=ax,
    )

    # ax.set_title(f"Performance Ratio - Task Type: {ttype}")
    ax.set_xlabel("Embedding Size", fontsize=25)
    ax.set_ylabel("Ratio of maximum performance", fontsize=25)
    ax.set_ylim(map_lim_task.get(ttype, 0.0), 1.0)

    apply_bar_hatch(ax)

    plt.tight_layout()

    filename = f"ratio_task_type_{sanitize_filename(ttype)}.pdf"
    fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
    plt.close(fig)